# 🧪 LLM Cost Optimization Lab
### Hands-on Python demos for all 4 context-window tax fixes

| Technique | Expected Savings |
|-----------|-----------------|
| 1. Semantic Caching | 40–80% of LLM calls skipped entirely |
| 2. Context Compression | 60–70% fewer tokens per call |
| 3. Stateful Context Management | Skip re-sending known context |
| 4. Token Budgets per Call Type | Hard cap on per-call spend |

> **Note:** This notebook uses the **new** Google Gen AI SDK (`google-genai`) for real LLM calls. Set your `GEMINI_API_KEY` before running Section 0.


## Section 0 — Setup & Shared Fixtures

In [6]:

!pip install google-genai sentence-transformers numpy tiktoken -q

import os, time, json, hashlib, re
from dataclasses import dataclass, field
from typing import Optional
import numpy as np
import tiktoken
from dotenv import load_dotenv
load_dotenv()

# ── Set your API key ──────────────────────────────────────────────────────────

from google import genai as google_genai
client = google_genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL  = "gemini-2.5-flash"   # fast, cheap, current stable model

# ── Token counter (tiktoken cl100k approximation) ─────────────────────────────
import tiktoken
_enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text: str) -> int:
    return len(_enc.encode(text))

# ── Shared fake agent context (mirrors the LinkedIn post's scenario) ──────────
SYSTEM_PROMPT = "You are a helpful sales analytics assistant with access to company data and tools."

TOOL_SCHEMAS = """
Available tools:
- get_weather(location: str) -> dict
- search_db(query: str, limit: int) -> list
- send_email(to: str, subject: str, body: str) -> bool
- create_ticket(title: str, priority: int, tags: list) -> str
- fetch_user(id: str) -> dict
- update_record(table: str, id: str, data: dict) -> bool
- run_query(sql: str) -> list
- generate_report(type: str, filters: dict) -> bytes
"""

CHAT_HISTORY = """
User: I need help analyzing Q3 sales data
Agent: Fetching Q3 sales data now...
Tool[fetch_user(id='u_123')] → {name: 'Alice', role: 'analyst'}
Tool[run_query(sql='SELECT * FROM sales WHERE quarter=3')] → 2847 rows
Agent: Retrieved 2847 sales records. Total revenue is $4.2M.
User: Break it down by region.
Agent: Running regional breakdown...
Tool[run_query(sql='SELECT region, SUM(revenue) FROM sales WHERE quarter=3 GROUP BY region')] → 5 rows
Agent: North $1.1M, South $0.9M, East $1.2M, West $0.7M, Central $0.3M
User: Flag any regions below 20% of total.
"""

RAG_CHUNKS = """
[Policy Doc] Regions below 20% of total revenue trigger automatic review per compliance section 4.2.
Historical data shows West region consistently underperforms in Q3 due to seasonal factors.
[Escalation Doc] When threshold is breached: notify regional VP within 2 hours, create HIGH-priority ticket,
include 3-quarter trend, CC finance@company.com and compliance@company.com.
[Glossary] Revenue threshold uses gross revenue excluding returns. Q3 = Jul-Sep.
"""

QUERY = "Flag regions below the revenue threshold and create alert tickets."

def build_naive_context() -> str:
    return "\n\n".join([SYSTEM_PROMPT, CHAT_HISTORY, RAG_CHUNKS, TOOL_SCHEMAS, QUERY])

naive_ctx   = build_naive_context()
naive_toks  = count_tokens(naive_ctx)
cost_per_1m = 0.10   # gemini-2.0-flash input $/1M tokens
naive_cost  = (naive_toks / 1_000_000) * cost_per_1m

print(f"Naive context : {naive_toks:,} tokens")
print(f"Cost per call : ${naive_cost:.5f}")
print(f"Cost @ 10K users × 30 calls/session: ${naive_cost * 30 * 10_000:,.2f}/day")


zsh:1: command not found: pip
Naive context : 416 tokens
Cost per call : $0.00004
Cost @ 10K users × 30 calls/session: $12.48/day


---
## Section 1 — Semantic Caching

**Idea:** Hash prompts *semantically* (not literally). If an incoming prompt is ≥ threshold similar to a cached entry, return the cached response — no LLM call needed.

```
Incoming prompt → embed → cosine_sim(cache) ≥ θ? → return cached response
                                                  NO → call LLM → store in cache
```


In [2]:

from sentence_transformers import SentenceTransformer
import numpy as np

# all-MiniLM-L6-v2 is fast but weak on paraphrases (scores ~0.65 for near-dups).
# all-mpnet-base-v2 is the recommended upgrade: same API, much better semantic
# similarity for short queries — paraphrases consistently score 0.82–0.92.
embedder = SentenceTransformer("all-mpnet-base-v2")   # 420 MB, still CPU-friendly

# ── In-memory semantic cache (swap dict for Redis in prod) ─────────────────────
@dataclass
class CacheEntry:
    prompt:    str
    response:  str
    embedding: np.ndarray
    tokens_saved: int = 0
    hits: int = 0

class SemanticCache:
    def __init__(self, threshold: float = 0.60):
        # ── Why 0.75? ────────────────────────────────────────────────────────
        # all-MiniLM-L6-v2 cosine similarities for paraphrases typically land
        # in the 0.75–0.88 range. 0.92 was too strict — only exact repeats
        # would hit. 0.75 catches genuine paraphrases while still rejecting
        # different-intent queries (which score ~0.4–0.6).
        # Rule of thumb: start at 0.75, raise if you see false positives.
        self.threshold = threshold
        self.entries: list[CacheEntry] = []

    def _embed(self, text: str) -> np.ndarray:
        return embedder.encode(text, normalize_embeddings=True)

    def get(self, prompt: str) -> tuple[Optional[CacheEntry], float]:
        """Returns (entry, best_similarity). Entry is None on miss."""
        if not self.entries:
            return None, 0.0
        emb   = self._embed(prompt)
        sims  = [float(np.dot(emb, e.embedding)) for e in self.entries]
        best_i = int(np.argmax(sims))
        best_sim = sims[best_i]
        if best_sim >= self.threshold:
            self.entries[best_i].hits += 1
            return self.entries[best_i], best_sim
        return None, best_sim

    def set(self, prompt: str, response: str, tokens_saved: int = 0):
        emb = self._embed(prompt)
        self.entries.append(CacheEntry(prompt, response, emb, tokens_saved))

    def stats(self):
        total_saved = sum(e.tokens_saved * e.hits for e in self.entries)
        total_cost_saved = (total_saved / 1_000_000) * cost_per_1m
        print(f"Cache entries : {len(self.entries)}")
        print(f"Total hits    : {sum(e.hits for e in self.entries)}")
        print(f"Tokens saved  : {total_saved:,}")
        print(f"Est. $ saved  : ${total_cost_saved:.5f}")
        print(f"\nThreshold used: {self.threshold} (tune up if false positives appear)")

cache = SemanticCache(threshold=0.60)

# ── Cached LLM call ────────────────────────────────────────────────────────────
def cached_llm_call(user_prompt: str, context: str) -> dict:
    hit, sim = cache.get(user_prompt)
    if hit:
        print(f"  ✅ CACHE HIT  (similarity={sim:.3f} ≥ {cache.threshold}) — skipped {count_tokens(context):,} tokens")
        return {"response": hit.response, "source": "cache", "tokens": 0}

    print(f"  ❌ CACHE MISS (best similarity={sim:.3f} < {cache.threshold}) — sending {count_tokens(context):,} tokens to LLM")
    t0 = time.time()
    from google.genai import types as gtypes
    msg = client.models.generate_content(
        model=MODEL,
        contents=context + "\n\n" + user_prompt,
        config=gtypes.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=512)
    )
    resp = msg.text
    toks = msg.usage_metadata.prompt_token_count
    cache.set(user_prompt, resp, tokens_saved=toks)
    print(f"  ⏱  LLM took {time.time()-t0:.2f}s | {toks:,} input tokens used")
    return {"response": resp, "source": "llm", "tokens": toks}

# ── Warm the cache with canonical query variants (simulates a real system
#    where you pre-populate with known frequent intents) ──────────────────────
print("Pre-warming cache with known query variants...")
WARM_QUERIES = [
    "Flag regions below the revenue threshold and create alert tickets.",
    "Identify underperforming regions and raise alerts.",
]
for wq in WARM_QUERIES:
    from google.genai import types as gtypes
    msg = client.models.generate_content(
        model=MODEL,
        contents=build_naive_context() + "\n\n" + wq,
        config=gtypes.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=512)
    )
    cache.set(wq, msg.text, tokens_saved=msg.usage_metadata.prompt_token_count)
    print(f"  Cached: '{wq[:60]}...'")

print(f"\nCache warmed with {len(cache.entries)} entries. Threshold = {cache.threshold}\n")

# ── Demo: 5 calls, some semantically similar ──────────────────────────────────
# Now these should all hit — the cache has both intent families pre-loaded
queries = [
    "Flag regions below the revenue threshold and create alert tickets.",        # exact HIT (warmed)
    "Which regions are under the revenue threshold? Create escalation tickets.", # paraphrase HIT
    "Identify underperforming regions and raise alerts.",                        # exact HIT (warmed)
    "What was the total Q3 revenue across all regions?",                         # MISS (different intent)
    "Flag regions below the revenue threshold and create alert tickets.",        # exact HIT
]

print("=== Semantic Cache Demo ===\n")
for i, q in enumerate(queries, 1):
    print(f"Call {i}: '{q[:60]}...'")
    result = cached_llm_call(q, build_naive_context())
    print(f"  Source: {result['source']}\n")

print("\n=== Cache Stats ===")
cache.stats()


/Users/poornimabyregowda/playground/blog_generator/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9556.19it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pre-warming cache with known query variants...
  Cached: 'Flag regions below the revenue threshold and create alert ti...'
  Cached: 'Identify underperforming regions and raise alerts....'

Cache warmed with 2 entries. Threshold = 0.6

=== Semantic Cache Demo ===

Call 1: 'Flag regions below the revenue threshold and create alert ti...'
  ✅ CACHE HIT  (similarity=1.000 ≥ 0.6) — skipped 416 tokens
  Source: cache

Call 2: 'Which regions are under the revenue threshold? Create escala...'
  ✅ CACHE HIT  (similarity=0.767 ≥ 0.6) — skipped 416 tokens
  Source: cache

Call 3: 'Identify underperforming regions and raise alerts....'
  ✅ CACHE HIT  (similarity=1.000 ≥ 0.6) — skipped 416 tokens
  Source: cache

Call 4: 'What was the total Q3 revenue across all regions?...'
  ❌ CACHE MISS (best similarity=0.345 < 0.6) — sending 416 tokens to LLM
  ⏱  LLM took 2.87s | 490 input tokens used
  Source: llm

Call 5: 'Flag regions below the revenue threshold and create alert ti...'
  ✅ CACHE HIT  (simi

---
## Section 2 — Context Compression (LLMLingua-style)

**Idea:** Not all tokens in your context are equally important. Compress by:
1. Scoring sentences by relevance to the current query
2. Keeping only the top-K% most informative sentences
3. Optionally preserving named entities (numbers, emails, tool names)

This is a lightweight approximation of [LLMLingua](https://github.com/microsoft/LLMLingua). For production use the real library.


In [5]:

import re
from sentence_transformers import SentenceTransformer, util

# Re-use embedder from Section 1

def extract_entities(text: str) -> list[str]:
    """Pull out tokens we must never drop."""
    patterns = [
        r'\$[\d\.]+[MKB]?',          # dollar amounts
        r'\b\d+%\b',                 # percentages
        r'\b[A-Z][a-z]+\b',           # proper nouns
        r'\w+@\w+\.\w+',            # emails
        r'\b(Q[1-4]|HIGH|LOW|MEDIUM)\b',  # domain keywords
    ]
    found = []
    for p in patterns:
        found.extend(re.findall(p, text))
    return list(set(found))

def compress_text(text: str, query: str, keep_ratio: float = 0.3,
                  preserve_entities: bool = True) -> str:
    """
    Score each sentence by cosine similarity to the query embedding.
    Keep top `keep_ratio` fraction; always preserve entity-bearing sentences.
    """
    sentences = [s.strip() for s in re.split(r'(?<=[.!?\n])\s*', text) if s.strip()]
    if len(sentences) <= 2:
        return text

    q_emb  = embedder.encode(query,     normalize_embeddings=True)
    s_embs = embedder.encode(sentences, normalize_embeddings=True)
    scores = util.cos_sim(q_emb, s_embs)[0].numpy()

    entities = extract_entities(text) if preserve_entities else []

    # Mark sentences that contain a preserved entity
    must_keep = set()
    if preserve_entities:
        for i, s in enumerate(sentences):
            if any(ent in s for ent in entities):
                must_keep.add(i)

    keep_n = max(1, int(len(sentences) * keep_ratio))
    ranked = sorted(range(len(sentences)), key=lambda i: scores[i], reverse=True)
    kept   = set(ranked[:keep_n]) | must_keep

    # Reconstruct in original order
    compressed = " ".join(sentences[i] for i in sorted(kept))
    return compressed

# ── Apply to each context component ──────────────────────────────────────────
print("=== Context Compression Demo ===\n")

components = {
    "CHAT_HISTORY" : CHAT_HISTORY,
    "RAG_CHUNKS"   : RAG_CHUNKS,
    "TOOL_SCHEMAS" : TOOL_SCHEMAS,
}

compressed_components = {}
for name, text in components.items():
    ratio = 0.3 if name != "TOOL_SCHEMAS" else 0.5  # keep more of tool schemas
    comp  = compress_text(text, QUERY, keep_ratio=ratio)
    orig_t = count_tokens(text)
    comp_t = count_tokens(comp)
    reduction = (1 - comp_t / orig_t) * 100
    compressed_components[name] = comp
    print(f"{name}")
    print(f"  Before : {orig_t:,} tokens")
    print(f"  After  : {comp_t:,} tokens  ({reduction:.1f}% reduction)")
    print(f"  Preview: {comp[:120]}...\n")

# ── Build compressed context ──────────────────────────────────────────────────
compressed_ctx  = "\n\n".join([
    SYSTEM_PROMPT,
    compressed_components["CHAT_HISTORY"],
    compressed_components["RAG_CHUNKS"],
    compressed_components["TOOL_SCHEMAS"],
    QUERY
])


orig_t = count_tokens(build_naive_context())
comp_t = count_tokens(compressed_ctx)
print(f"Total context   : {orig_t:,} → {comp_t:,} tokens  ({(1-comp_t/orig_t)*100:.1f}% reduction)")
print(f"Cost per call   : ${(orig_t/1e6)*cost_per_1m:.5f} → ${(comp_t/1e6)*cost_per_1m:.5f}")

# ── Real LLM call with compressed context ─────────────────────────────────────
print("\nCalling LLM with compressed context...")
from google.genai import types as gtypes
msg = client.models.generate_content(
    model=MODEL,
    contents=compressed_ctx,
    config=gtypes.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=512)
)
print(f"\nResponse ({msg.usage_metadata.prompt_token_count:,} actual input tokens):")
print(msg.text)


=== Context Compression Demo ===

CHAT_HISTORY
  Before : 175 tokens
  After  : 169 tokens  (3.4% reduction)
  Preview: User: I need help analyzing Q3 sales data Agent: Fetching Q3 sales data now. Tool[fetch_user(id='u_123')] → {name: 'Alic...

RAG_CHUNKS
  Before : 102 tokens
  After  : 82 tokens  (19.6% reduction)
  Preview: [Policy Doc] Regions below 20% of total revenue trigger automatic review per compliance section 4. Historical data shows...

TOOL_SCHEMAS
  Before : 116 tokens
  After  : 50 tokens  (56.9% reduction)
  Preview: Available tools: - search_db(query: str, limit: int) -> list - send_email(to: str, subject: str, body: str) -> bool - cr...

Total context   : 416 → 328 tokens  (21.2% reduction)
Cost per call   : $0.00004 → $0.00003

Calling LLM with compressed context...

Response (367 actual input tokens):
Okay, let's calculate the threshold and flag regions.

Total Q3 Revenue: $4M


---
## Section 3 — Stateful Context Management (LangGraph-style checkpointing)

**Idea:** Track what the agent already "knows" in a session. After the first call, skip re-sending static context (tool schemas, system instructions, user profile) that hasn't changed.

```
Call 1:  [system] + [tools] + [user profile] + [history] + [query]   ← full
Call 2:  [delta history] + [query]                                    ← skip known static chunks
Call N:  [delta history] + [query]                                    ← skip known static chunks
```


In [7]:

import hashlib

@dataclass
class ContextChunk:
    name:    str
    content: str
    static:  bool = True      # static = safe to skip after first send
    checksum: str = field(init=False)

    def __post_init__(self):
        self.checksum = hashlib.md5(self.content.encode()).hexdigest()[:8]

class StatefulContextManager:
    def __init__(self):
        self.sent_checksums: set[str] = set()
        self.call_count = 0
        self.total_tokens_sent  = 0
        self.total_tokens_saved = 0

    def build_context(self, chunks: list[ContextChunk]) -> tuple[str, dict]:
        self.call_count += 1
        included, skipped = [], []

        for chunk in chunks:
            if chunk.static and chunk.checksum in self.sent_checksums:
                skipped.append(chunk)
                self.total_tokens_saved += count_tokens(chunk.content)
            else:
                included.append(chunk)
                self.sent_checksums.add(chunk.checksum)
                self.total_tokens_sent += count_tokens(chunk.content)

        context = "\n\n".join(c.content for c in included)
        meta = {
            "call": self.call_count,
            "included": [c.name for c in included],
            "skipped":  [c.name for c in skipped],
            "tokens_this_call": count_tokens(context),
        }
        return context, meta

    def stats(self):
        total = self.total_tokens_sent + self.total_tokens_saved
        pct_saved = (self.total_tokens_saved / total * 100) if total else 0
        saved_cost = (self.total_tokens_saved / 1e6) * cost_per_1m
        print(f"Calls made     : {self.call_count}")
        print(f"Tokens sent    : {self.total_tokens_sent:,}")
        print(f"Tokens saved   : {self.total_tokens_saved:,}  ({pct_saved:.1f}%)")
        print(f"Money saved    : ${saved_cost:.4f}")

# ── Simulate 5 turns of an agent session ─────────────────────────────────────
manager = StatefulContextManager()

# Static chunks (sent once, never again)
static_chunks = [
    ContextChunk("system_prompt",  SYSTEM_PROMPT,  static=True),
    ContextChunk("tool_schemas",   TOOL_SCHEMAS,   static=True),
    ContextChunk("rag_chunks",     RAG_CHUNKS,     static=True),
]

session_queries = [
    "Flag regions below the revenue threshold.",
    "Create HIGH-priority tickets for West and Central regions.",
    "Draft an alert email to the regional VPs.",
    "Summarize the escalation actions taken this session.",
    "What is the total revenue for flagged regions?",
]

print("=== Stateful Context Demo ===\n")
responses = []
for turn, query in enumerate(session_queries, 1):
    # Dynamic chunk: growing history + current query (always re-sent)
    history_so_far = CHAT_HISTORY + "\n" + "\n".join(
        f"Turn {i}: {q}" for i, q in enumerate(session_queries[:turn-1], 1)
    )
    dynamic_chunks = [
        ContextChunk("chat_history", history_so_far, static=False),
        ContextChunk("current_query", query,         static=False),
    ]

    context, meta = manager.build_context(static_chunks + dynamic_chunks)

    print(f"Turn {turn} | query: '{query[:50]}...'")
    print(f"  Included : {meta['included']}")
    print(f"  Skipped  : {meta['skipped']}")
    print(f"  Tokens   : {meta['tokens_this_call']:,}\n")

    from google.genai import types as gtypes
    msg = client.models.generate_content(
        model=MODEL,
        contents=context,
        config=gtypes.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=256)
    )
    responses.append(msg.text)

print("\n=== Session Stats ===")
manager.stats()
print("\nFinal response:")
print(responses[-1])


=== Stateful Context Demo ===

Turn 1 | query: 'Flag regions below the revenue threshold....'
  Included : ['system_prompt', 'tool_schemas', 'rag_chunks', 'chat_history', 'current_query']
  Skipped  : []
  Tokens   : 412

Turn 2 | query: 'Create HIGH-priority tickets for West and Central ...'
  Included : ['chat_history', 'current_query']
  Skipped  : ['system_prompt', 'tool_schemas', 'rag_chunks']
  Tokens   : 197

Turn 3 | query: 'Draft an alert email to the regional VPs....'
  Included : ['chat_history', 'current_query']
  Skipped  : ['system_prompt', 'tool_schemas', 'rag_chunks']
  Tokens   : 211

Turn 4 | query: 'Summarize the escalation actions taken this sessio...'
  Included : ['chat_history', 'current_query']
  Skipped  : ['system_prompt', 'tool_schemas', 'rag_chunks']
  Tokens   : 225

Turn 5 | query: 'What is the total revenue for flagged regions?...'
  Included : ['chat_history', 'current_query']
  Skipped  : ['system_prompt', 'tool_schemas', 'rag_chunks']
  Tokens   : 238


---
## Section 4 — Token Budgets per Call Type

**Idea:** Not all LLM calls need the same context. Enforce hard caps:

| Call Type | Context Budget | What to Include |
|-----------|---------------|-----------------|
| Tool decision call | 2 K tokens | System + tool schemas + current query |
| Mid-session reasoning | 4 K tokens | Above + compressed history |
| Final response | 8 K tokens | Full compressed context |

Use LiteLLM or a custom truncator to enforce these.


In [8]:

from enum import Enum

class CallType(Enum):
    TOOL_DECISION  = ("tool_decision",  2_000)
    MID_REASONING  = ("mid_reasoning",  4_000)
    FINAL_RESPONSE = ("final_response", 8_000)

    def __init__(self, label, budget):
        self.label  = label
        self.budget = budget   # max input tokens

def truncate_to_budget(text: str, budget: int, preserve_end: bool = True) -> str:
    """
    Trim text to fit within `budget` tokens.
    If preserve_end=True, keep the END of the text (most recent = most relevant).
    """
    tokens = _enc.encode(text)
    if len(tokens) <= budget:
        return text
    if preserve_end:
        tokens = tokens[-budget:]
    else:
        tokens = tokens[:budget]
    return _enc.decode(tokens)

def build_budgeted_context(call_type: CallType, query: str,
                            history: str = "", rag: str = "") -> tuple[str, int, int]:
    """
    Assemble context according to call-type budget.
    Returns (context_str, tokens_used, tokens_saved_vs_naive).
    """
    budget = call_type.budget

    if call_type == CallType.TOOL_DECISION:
        ctx = "\n\n".join([SYSTEM_PROMPT, TOOL_SCHEMAS, query])

    elif call_type == CallType.MID_REASONING:
        comp_history = compress_text(history, query, keep_ratio=0.4) if history else ""
        ctx = "\n\n".join(filter(None, [SYSTEM_PROMPT, TOOL_SCHEMAS, comp_history, query]))

    else:  # FINAL_RESPONSE
        comp_history = compress_text(history, query, keep_ratio=0.5) if history else ""
        comp_rag     = compress_text(rag,     query, keep_ratio=0.4) if rag     else ""
        ctx = "\n\n".join(filter(None, [SYSTEM_PROMPT, comp_rag, comp_history, query]))

    ctx_truncated = truncate_to_budget(ctx, budget, preserve_end=True)
    toks_used     = count_tokens(ctx_truncated)
    toks_naive    = count_tokens(build_naive_context())
    toks_saved    = toks_naive - toks_used

    return ctx_truncated, toks_used, toks_saved

# ── Demo: one query through all 3 call types ─────────────────────────────────
print("=== Token Budget Demo ===\n")
total_naive = 0
total_budgeted = 0

for ct in CallType:
    ctx, used, saved = build_budgeted_context(ct, QUERY, CHAT_HISTORY, RAG_CHUNKS)
    naive = count_tokens(build_naive_context())
    total_naive    += naive
    total_budgeted += used
    pct = (1 - used / naive) * 100

    print(f"CallType: {ct.label:20s}  budget={ct.budget:,}")
    print(f"  Naive tokens  : {naive:,}")
    print(f"  Budgeted      : {used:,}  ({pct:.1f}% reduction)")
    print(f"  Saved         : {saved:,} tokens  (${(saved/1e6)*cost_per_1m:.5f})")
    print()

# ── Call LLM with budgeted context ───────────────────────────────────────────
print("Calling LLM with FINAL_RESPONSE budget context...")
ctx, used, _ = build_budgeted_context(CallType.FINAL_RESPONSE, QUERY, CHAT_HISTORY, RAG_CHUNKS)
from google.genai import types as gtypes
msg = client.models.generate_content(
    model=MODEL,
    contents=ctx,
    config=gtypes.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=512)
)
print(f"Actual input tokens (API): {msg.usage_metadata.prompt_token_count:,}")
print(f"\nResponse:\n{msg.text}")


=== Token Budget Demo ===

CallType: tool_decision         budget=2,000
  Naive tokens  : 416
  Budgeted      : 141  (66.1% reduction)
  Saved         : 275 tokens  ($0.00003)

CallType: mid_reasoning         budget=4,000
  Naive tokens  : 416
  Budgeted      : 310  (25.5% reduction)
  Saved         : 106 tokens  ($0.00001)

CallType: final_response        budget=8,000
  Naive tokens  : 416
  Budgeted      : 277  (33.4% reduction)
  Saved         : 139 tokens  ($0.00001)

Calling LLM with FINAL_RESPONSE budget context...
Actual input tokens (API): 310

Response:
Okay, let's calculate the threshold and identify underperforming regions.

**1. Calculate Total Revenue:**
Total Q3 revenue = $1.1M (North) + $0.9M (South) + $1.2M (East) + $0.7M (West) + $1.0M (Central) = **$4.9M**

**2. Calculate 20% Revenue Threshold:**
20% of $4.9M = **$0.98M**

**3. Flag Regions Below Threshold:**

*   **North:** $1.1M (Above threshold)
*   **South:** $0.9M (Below threshold)
*   **East:** $1.2M (Above thre

---
## Section 5 — 🚀 Combined Optimization Pipeline

Stack all four techniques together for maximum savings.

```
Query → SemanticCache? ──YES──→ return cached response
            │
           NO
            ↓
     compress context (LLMLingua-style)
            ↓
     stateful manager (skip known chunks)
            ↓
     enforce token budget (by call type)
            ↓
     LLM call → cache result → return
```


In [9]:

# Re-use objects from sections above (cache, manager)
pipeline_cache   = SemanticCache(threshold=0.60)
pipeline_manager = StatefulContextManager()

static_chunks_p = [
    ContextChunk("system_prompt",  SYSTEM_PROMPT,  static=True),
    ContextChunk("tool_schemas",   TOOL_SCHEMAS,   static=True),
]

def optimized_agent_call(query: str, history: str, rag: str,
                          call_type: CallType = CallType.FINAL_RESPONSE) -> dict:
    naive_toks = count_tokens(build_naive_context())

    # 1. Semantic cache check
    hit, sim = pipeline_cache.get(query)
    if hit:
        print(f"  ⚡ CACHE HIT (sim={sim:.3f}) — saved {naive_toks:,} tokens entirely!")
        return {"response": hit.response, "tokens_used": 0,
                "tokens_saved": naive_toks, "source": "cache"}

    # 2. Compress dynamic context
    comp_history = compress_text(history, query, keep_ratio=0.4)
    comp_rag     = compress_text(rag,     query, keep_ratio=0.35)

    # 3. Stateful manager (skip static chunks already sent)
    rag_chunk  = ContextChunk("rag",     comp_rag,     static=False)
    hist_chunk = ContextChunk("history", comp_history, static=False)
    q_chunk    = ContextChunk("query",   query,        static=False)
    ctx_str, meta = pipeline_manager.build_context(
        static_chunks_p + [rag_chunk, hist_chunk, q_chunk])

    # 4. Token budget enforcement
    ctx_str = truncate_to_budget(ctx_str, call_type.budget, preserve_end=True)
    toks_used = count_tokens(ctx_str)

    print(f"  📊 Naive {naive_toks:,} → Optimized {toks_used:,} tokens "
          f"({(1-toks_used/naive_toks)*100:.1f}% saved)")
    print(f"  Skipped chunks: {meta['skipped']}")

    # 5. LLM call
    from google.genai import types as gtypes
    msg = client.models.generate_content(
        model=MODEL,
        contents=ctx_str,
        config=gtypes.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=512)
    )
    response = msg.text
    pipeline_cache.set(query, response, tokens_saved=toks_used)

    return {
        "response":     response,
        "tokens_used":  msg.usage_metadata.prompt_token_count,
        "tokens_saved": naive_toks - toks_used,
        "source":       "llm"
    }

# ── Run 6 calls ───────────────────────────────────────────────────────────────
test_queries = [
    ("Flag regions below the 20% threshold.",       CallType.TOOL_DECISION),
    ("Create tickets for West and Central regions.", CallType.MID_REASONING),
    ("Flag regions below the 20% threshold.",       CallType.TOOL_DECISION),  # cache hit
    ("Draft the VP alert email.",                   CallType.FINAL_RESPONSE),
    ("Which regions are under revenue threshold?",  CallType.TOOL_DECISION),  # near-dup → cache
    ("Total revenue for Q3?",                       CallType.FINAL_RESPONSE),
]

print("=== Combined Pipeline Demo ===\n")
total_naive_session    = 0
total_optimized_session = 0

for i, (q, ct) in enumerate(test_queries, 1):
    naive = count_tokens(build_naive_context())
    total_naive_session += naive
    print(f"Call {i} [{ct.label}]: '{q}'")
    result = optimized_agent_call(q, CHAT_HISTORY, RAG_CHUNKS, ct)
    total_optimized_session += result["tokens_used"]
    print()

# ── Final report ──────────────────────────────────────────────────────────────
overall_reduction = (1 - total_optimized_session / total_naive_session) * 100
cost_naive    = (total_naive_session    / 1e6) * cost_per_1m
cost_optimized = (total_optimized_session / 1e6) * cost_per_1m

print("=" * 50)
print("SESSION SUMMARY")
print("=" * 50)
print(f"Total tokens  (naive)     : {total_naive_session:,}")
print(f"Total tokens  (optimized) : {total_optimized_session:,}")
print(f"Overall reduction         : {overall_reduction:.1f}%")
print(f"Session cost  (naive)     : ${cost_naive:.4f}")
print(f"Session cost  (optimized) : ${cost_optimized:.4f}")
print(f"Savings per session       : ${cost_naive - cost_optimized:.4f}")
print(f"Savings @ 10K users/day   : ${(cost_naive - cost_optimized) * 10_000:,.2f}")
print()
print("Pipeline cache stats:")
pipeline_cache.stats()


=== Combined Pipeline Demo ===

Call 1 [tool_decision]: 'Flag regions below the 20% threshold.'
  📊 Naive 416 → Optimized 390 tokens (6.2% saved)
  Skipped chunks: []

Call 2 [mid_reasoning]: 'Create tickets for West and Central regions.'
  📊 Naive 416 → Optimized 259 tokens (37.7% saved)
  Skipped chunks: ['system_prompt', 'tool_schemas']

Call 3 [tool_decision]: 'Flag regions below the 20% threshold.'
  ⚡ CACHE HIT (sim=1.000) — saved 416 tokens entirely!

Call 4 [final_response]: 'Draft the VP alert email.'
  📊 Naive 416 → Optimized 267 tokens (35.8% saved)
  Skipped chunks: ['system_prompt', 'tool_schemas']

Call 5 [tool_decision]: 'Which regions are under revenue threshold?'
  📊 Naive 416 → Optimized 258 tokens (38.0% saved)
  Skipped chunks: ['system_prompt', 'tool_schemas']

Call 6 [final_response]: 'Total revenue for Q3?'
  📊 Naive 416 → Optimized 271 tokens (34.9% saved)
  Skipped chunks: ['system_prompt', 'tool_schemas']

SESSION SUMMARY
Total tokens  (naive)     : 2,496
Tota